In [1]:
# Import necessary libraries
from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "amex-46887"
DATASET_ID = "amex_risk"

client = bigquery.Client(project=PROJECT_ID)

In [2]:
# List tables in the dataset
query = f"""
SELECT table_name
FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name
"""

client.query(query).to_dataframe()

,table_name
0,customer_features_base
1,customer_features_latest
2,customer_features_v2
3,model_dataset_v1
4,sample_submission_raw
5,test_data_raw
6,train_data_raw
7,train_joined
8,train_labels_raw


In [3]:
# Check the number of rows in the train_data_raw table
query = f"""
SELECT COUNT(*) AS n_rows
FROM `{PROJECT_ID}.{DATASET_ID}.train_data_raw`
"""
client.query(query).to_dataframe()

,n_rows
0,5531451


In [4]:
# Unique Customers
query = f"""
SELECT COUNT(DISTINCT customer_ID) AS unique_customers
FROM `{PROJECT_ID}.{DATASET_ID}.train_data_raw`
"""
client.query(query).to_dataframe()

,unique_customers
0,458913


In [5]:
# Label Balance
query = f"""
SELECT target, COUNT(*) AS n_customers
FROM `{PROJECT_ID}.{DATASET_ID}.train_labels_raw`
GROUP BY target
ORDER BY target
"""
client.query(query).to_dataframe()

,target,n_customers
0,0,340085
1,1,118828


In [6]:
# Missing label customers
query = f"""
SELECT COUNT(*) AS missing_label_customers
FROM `{PROJECT_ID}.{DATASET_ID}.train_labels_raw` l
LEFT JOIN (
  SELECT DISTINCT customer_ID
  FROM `{PROJECT_ID}.{DATASET_ID}.train_data_raw`
) t
USING (customer_ID)
WHERE t.customer_ID IS NULL
"""
client.query(query).to_dataframe()

,missing_label_customers
0,0


In [7]:
# Distribution of number of statements per customer
query = f"""
WITH statement_counts AS (
  SELECT customer_ID, COUNT(*) AS n_statements
  FROM `{PROJECT_ID}.{DATASET_ID}.train_data_raw`
  GROUP BY customer_ID
)
SELECT n_statements, COUNT(*) AS n_customers
FROM statement_counts
GROUP BY n_statements
ORDER BY n_statements
"""
client.query(query).to_dataframe()

,n_statements,n_customers
0,1,5120
1,2,6098
2,3,5778
3,4,4673
4,5,4671
5,6,5515
6,7,5198
7,8,6110
8,9,6411
9,10,6721


In [8]:
# Statements per customer statistics
query = f"""
WITH statement_counts AS (
  SELECT customer_ID, COUNT(*) AS n_statements
  FROM `{PROJECT_ID}.{DATASET_ID}.train_data_raw`
  GROUP BY customer_ID
)
SELECT
  MIN(n_statements) AS min_statements,
  MAX(n_statements) AS max_statements,
  AVG(n_statements) AS avg_statements
FROM statement_counts
"""
client.query(query).to_dataframe()

,min_statements,max_statements,avg_statements
0,1,13,12.053376


## Observations

- `train_data_raw` contains 5,531,451 rows and 458,913 unique customers.
- `train_labels_raw` contains 458,913 labelled customers.
- The target is imbalanced, with more non-default cases than default cases.
- There are no labelled customers missing from the raw transaction table.
- Customers have between 1 and 13 monthly statements, with an average of about 12.05.
- This confirms the dataset has a customer-level sequential structure suitable for aggregated and temporal feature engineering.

In [9]:
from pathlib import Path

sql_path = Path("../sql/04_latest_record_features.sql")
query = sql_path.read_text()

client.query(query).result()
print("customer_features_latest created successfully.")

customer_features_latest created successfully.


In [10]:
query = """
SELECT
  COUNT(*) AS row_count,
  COUNT(DISTINCT customer_ID) AS customer_count
FROM `amex-46887.amex_risk.customer_features_base`
"""
client.query(query).to_dataframe()

,row_count,customer_count
0,458913,458913


In [11]:
query = """
SELECT *
FROM `amex-46887.amex_risk.customer_features_base`
LIMIT 5
"""
client.query(query).to_dataframe()

,customer_ID,target,n_statements,P_2_avg,P_2_min,P_2_max,P_2_std,P_2_nulls,D_39_avg,D_39_min,...,B_11_avg,B_11_min,B_11_max,B_11_std,B_11_nulls,S_3_avg,S_3_min,S_3_max,S_3_std,S_3_nulls
0,ca59e69a8c5a2098d0454e3d7654ec3492382d3c9d8246...,0,1,0.798627,0.798627,0.798627,NaN,0,0.002210,0.002210,...,0.008086,0.008086,0.008086,NaN,0,0.244250,0.244250,0.244250,NaN,0
1,4192014dc362945085be20ca281f61477d98bfecd1bfc3...,1,1,0.581358,0.581358,0.581358,NaN,0,0.007240,0.007240,...,0.004918,0.004918,0.004918,NaN,0,0.170583,0.170583,0.170583,NaN,0
2,7d72d5d9ea9fec3e6ad659894dae23f6a05afbafba2863...,1,1,0.532148,0.532148,0.532148,NaN,0,0.005291,0.005291,...,0.027639,0.027639,0.027639,NaN,0,0.166451,0.166451,0.166451,NaN,0
3,6ec187f959f3d6ba74c4ed396eec4e54803b1cd79e81dc...,0,1,0.882992,0.882992,0.882992,NaN,0,0.123693,0.123693,...,1.338380,1.338380,1.338380,NaN,0,-0.007232,-0.007232,-0.007232,NaN,0
4,255cbf95bd80fe15f4603cebaee56559b8b1a63acc85ab...,1,1,0.219865,0.219865,0.219865,NaN,0,0.090718,0.090718,...,0.851665,0.851665,0.851665,NaN,0,0.138109,0.138109,0.138109,NaN,0


In [12]:
from pathlib import Path

query = Path("../sql/03_base_customer_features.sql").read_text()
client.query(query).result()
print("customer_features_base recreated successfully.")

customer_features_base recreated successfully.


In [13]:
query = """
SELECT *
FROM `amex-46887.amex_risk.customer_features_base`
LIMIT 1
"""
df_check = client.query(query).to_dataframe()
df_check.columns.tolist()

['customer_ID',
 'target',
 'n_statements',
 'P_2_avg',
 'P_2_min',
 'P_2_max',
 'P_2_std',
 'P_2_nulls',
 'D_39_avg',
 'D_39_min',
 'D_39_max',
 'D_39_std',
 'D_39_nulls',
 'B_9_avg',
 'B_9_min',
 'B_9_max',
 'B_9_std',
 'B_9_nulls',
 'R_1_avg',
 'R_1_min',
 'R_1_max',
 'R_1_std',
 'R_1_nulls',
 'B_11_avg',
 'B_11_min',
 'B_11_max',
 'B_11_std',
 'B_11_nulls',
 'S_3_avg',
 'S_3_min',
 'S_3_max',
 'S_3_std',
 'S_3_nulls']

In [14]:
from pathlib import Path

for sql_file in [
    "../sql/04_latest_record_features.sql",
    "../sql/05_model_dataset.sql",
]:
    query = Path(sql_file).read_text()
    client.query(query).result()
    print(f"Finished: {sql_file}")

Finished: ../sql/04_latest_record_features.sql
Finished: ../sql/05_model_dataset.sql


In [15]:
query = """
SELECT
  COUNT(*) AS row_count,
  COUNT(DISTINCT customer_ID) AS customer_count
FROM `amex-46887.amex_risk.model_dataset_v1`
"""
client.query(query).to_dataframe()

,row_count,customer_count
0,458913,458913


In [16]:
query = """
SELECT *
FROM `amex-46887.amex_risk.model_dataset_v1`
LIMIT 5
"""
client.query(query).to_dataframe()

,customer_ID,target,n_statements,P_2_avg,P_2_min,P_2_max,P_2_std,P_2_nulls,D_39_avg,D_39_min,...,B_9_delta,R_1_first,R_1_last,R_1_delta,B_11_first,B_11_last,B_11_delta,S_3_first,S_3_last,S_3_delta
0,cc70ddc2b2d4beef767731d2fc9ebbc547000ebefbeea3...,1,1,0.433507,0.433507,0.433507,NaN,0,0.004427,0.004427,...,0.0,0.006162,0.006162,0.0,0.015095,0.015095,0.0,0.185926,0.185926,0.0
1,d0f09a99a27455d230531d39ffacb2f262d7d613be9c3d...,0,1,0.572998,0.572998,0.572998,NaN,0,0.148708,0.148708,...,0.0,0.008399,0.008399,0.0,0.060093,0.060093,0.0,0.170137,0.170137,0.0
2,4cf080cd01effa1f9ac5da341277a4a5777d03a5dba64b...,1,1,0.540081,0.540081,0.540081,NaN,0,0.007145,0.007145,...,0.0,0.007934,0.007934,0.0,0.006573,0.006573,0.0,0.352069,0.352069,0.0
3,feec993e93dc9d26fbae5a291ab2bd97e76c2407ef159f...,0,1,0.313847,0.313847,0.313847,NaN,0,0.211802,0.211802,...,0.0,0.009279,0.009279,0.0,0.872583,0.872583,0.0,0.202343,0.202343,0.0
4,02fdef39c524af89e22c737c397a321f3abe71a5a0d991...,1,1,0.515970,0.515970,0.515970,NaN,0,0.002451,0.002451,...,0.0,0.001580,0.001580,0.0,0.029790,0.029790,0.0,0.310393,0.310393,0.0


In [17]:
query = """
SELECT
  target,
  COUNT(*) AS n_customers
FROM `amex-46887.amex_risk.model_dataset_v1`
GROUP BY target
ORDER BY target
"""
client.query(query).to_dataframe()

,target,n_customers
0,0,340085
1,1,118828
